In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import numpy as np
import copy

# ========================== DEVICE & CONSTANTS ==========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ⚠️ ИСПРАВЛЕНИЕ 1: Увеличить NUM_WORKERS для параллельной загрузки данных
NUM_WORKERS = 0
BATCH_SIZE = 32  # Увеличен благодаря AMP и Gradient Accumulation
GRADIENT_ACCUMULATION_STEPS = 2  # Симулируем батч размером 128 (64*2)
TARGET_BALANCE_RATIO = 1.0

print(f"Workers: {NUM_WORKERS}, Batch Size: {BATCH_SIZE}, "
      f"Gradient Accumulation: {GRADIENT_ACCUMULATION_STEPS}, "
      f"Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

# ========================== DATASET CLASS ==========================
class DeepfakeDataset(Dataset):
    def __init__(self, img_dir, ids_list, labels=None, transform=None):
        self.img_dir = img_dir
        self.labels = labels
        self.transform = transform
        self.files_ids = ids_list

    def __len__(self):
        return len(self.files_ids)

    def __getitem__(self, idx):
        img_id = self.files_ids[idx]
        fname = f"{img_id}.jpg"
        img_path = os.path.join(self.img_dir, fname)

        if not os.path.exists(img_path):
            print(f"Warning: Image file not found, skipping: {img_path}")
            # Возвращаем плейсхолдер, который потом отфильтруется
            return None, None

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image ID {img_id}: {e}")
            return None, None

        if self.transform:
            img = self.transform(img)
        
        label = self.labels.get(img_id, -1)

        return img, label

# Функция для отфильтровывания "плохих" сэмплов (None), которые могли появиться из-за ошибок загрузки
def collate_fn(batch):
    batch = list(filter(lambda x: x[0] is not None, batch))
    if not batch:
        return torch.Tensor(), torch.Tensor()
    return torch.utils.data.dataloader.default_collate(batch)

# ========================== TRANSFORMS ==========================
mean = torch.tensor([0.5192, 0.4276, 0.3844])
std = torch.tensor([0.2860, 0.2640, 0.2635])

transform_rule = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean.tolist(), std=std.tolist())
])

# ========================== LOAD DATA & RESAMPLING ==========================
train_dir = "dataset/train_images"
csv_dir = "dataset/train_solution.csv"

labels = {}
with open(csv_dir, "r") as file:
    for line in file:
        line = line.strip()
        if "," in line:
            idx, val = line.split(",")
            if idx.isdigit() and val.isdigit():
                 labels[int(idx)] = int(val)

all_ids = list(labels.keys())
all_labels_stratify = [labels[i] for i in all_ids]

train_ids_unique, val_ids = train_test_split(
    all_ids, test_size=0.2, random_state=42, stratify=all_labels_stratify
)

count_real = sum(1 for i in all_ids if labels[i] == 0)
count_fake = sum(1 for i in all_ids if labels[i] == 1)

total_samples = len(all_ids)
# Добавляем 1e-6 для избежания деления на ноль, если какого-то класса нет
weight_real = total_samples / (2 * (count_real + 1e-6))
weight_fake = total_samples / (2 * (count_fake + 1e-6))

class_weights = torch.tensor([weight_real, weight_fake]).float().to(DEVICE)

print("\n" + "="*50)
print("📊 Анализ дисбаланса и веса классов:")
print(f"  Всего образцов: {total_samples}")
print(f"  Реальные лица (Класс 0): {count_real} -> Weight: {weight_real:.2f}")
print(f"  Синтетические лица (Класс 1): {count_fake} -> Weight: {weight_fake:.2f}")

# Oversampling для обучающей выборки
real_count_train = sum(1 for i in train_ids_unique if labels[i] == 0)
fake_count_train = sum(1 for i in train_ids_unique if labels[i] == 1)

balanced_train_ids = list(train_ids_unique)
if fake_count_train > 0:
    target_fake_count = int(real_count_train * TARGET_BALANCE_RATIO)
    oversample_multiplier = target_fake_count // fake_count_train
    
    print("\n>>> Проведение Oversampling для Fake класса...")
    fake_ids_to_oversample = [id for id in train_ids_unique if labels[id] == 1]
    
    for _ in range(oversample_multiplier - 1):
        balanced_train_ids.extend(fake_ids_to_oversample)
    
    # Добавляем остаток, если нужно
    remainder = target_fake_count % fake_count_train
    balanced_train_ids.extend(np.random.choice(fake_ids_to_oversample, remainder, replace=False))


    print("-" * 50)
    print(f"Исходный размер обучающего набора: {len(train_ids_unique)} (R={real_count_train}, F={fake_count_train})")
    final_fake_count = sum(1 for i in balanced_train_ids if labels[i] == 1)
    print(f"Балансированный размер обучающего набора: {len(balanced_train_ids)} (R={real_count_train}, F={final_fake_count}). Соотношение F/R ≈ {final_fake_count/real_count_train:.2f}")
else:
    print("В обучающей выборке отсутствуют фейки, oversampling не требуется.")


# ========================== DATASETS ==========================
train_dataset = DeepfakeDataset(train_dir, ids_list=balanced_train_ids, labels=labels, transform=transform_rule)
val_dataset = DeepfakeDataset(train_dir, ids_list=val_ids, labels=labels, transform=transform_rule)

# ⚠️ ИСПРАВЛЕНИЕ 2: Оптимизированные DataLoaders с pin_memory, workers и collate_fn
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,  # ✅ Ускоряет передачу данных на GPU
    persistent_workers=NUM_WORKERS > 0,
    drop_last=True, # ✅ Отбрасываем последний неполный батч для стабильности
    collate_fn=collate_fn # ✅ Для обработки ошибок загрузки
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_fn
)

print(f"\nDataLoaders created successfully! Workers: {NUM_WORKERS}")
print(f"Train batches (Balanced): {len(train_loader)}, Val batches: {len(val_loader)}")

# ========================== MODEL ARCHITECTURE ==========================
class DeepBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c)
        ) if in_c != out_c else nn.Identity() # nn.Identity() эффективнее
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        return self.relu(self.conv(x) + self.shortcut(x))

class DeepfakeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Уменьшим сложность для более быстрой сходимости и меньшего риска переобучения
        self.initial_conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1, stride=2, bias=False), # stride=2 уменьшит размерность
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )
        self.layer1 = DeepBlock(64, 128)
        self.layer2 = DeepBlock(128, 256)
        self.layer3 = DeepBlock(256, 512)
        self.layer4 = DeepBlock(512, 512) # Уменьшено с 1024
        
        self.pool = nn.MaxPool2d(2) # MaxPool может быть лучше для классификации
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 2) # Выходной слой соответствует последнему блоку
        )

    def forward(self, x):
        x = self.initial_conv(x) # -> 128x128
        x = self.pool(self.layer1(x)) # -> 64x64
        x = self.pool(self.layer2(x)) # -> 32x32
        x = self.pool(self.layer3(x)) # -> 16x16
        x = self.pool(self.layer4(x)) # -> 8x8
        x = self.global_pool(x)
        return self.classifier(x)

# ========================== TRAINING FUNCTION (ПОЛНОСТЬЮ ПЕРЕРАБОТАНА) ==========================
def train_model(model, train_loader, val_loader, class_weights, num_epochs=70):
    # Loss, Optimizer, Scheduler
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )
    
    # Инструмент для смешанной точности (AMP)
    scaler = GradScaler()
    
    best_f1 = 0.0
    patience_counter = 0
    best_model_state = None
    MODEL_SAVE_DIR = "models"
    os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

    for epoch in range(num_epochs):
        # ==================== ЭТАП ОБУЧЕНИЯ (TRAIN) ====================
        model.train()
        train_loss = 0.0
        train_preds, train_true = [], []
        
        # ✅ Правильный цикл по DataLoader с прогресс-баром
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{num_epochs} [Train]", leave=False)
        optimizer.zero_grad() # ✅ Очищаем градиенты в начале эпохи (важно для accumulation)
        
        for step, (images, labels) in enumerate(pbar):
            if images.numel() == 0: continue # Пропускаем пустые батчи из-за ошибок загрузки
            
            # Перемещаем батч на GPU
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            
            # ✅ Automatic Mixed Precision: вычисления в float16
            with autocast(dtype=torch.float16):
                outputs = model(images)
                # ✅ Нормализуем loss для градиентной аккумуляции
                loss = criterion(outputs, labels) / GRADIENT_ACCUMULATION_STEPS
            
            # ✅ Масштабируем loss и делаем backward pass
            scaler.scale(loss).backward()
            
            # ✅ Шаг оптимизатора и обновление весов каждые N шагов
            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                # scaler.unscale_ нужен для clip_grad_norm
                scaler.unscale_(optimizer) 
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                scaler.step(optimizer) # Делает шаг оптимизатора
                scaler.update() # Обновляет масштаб
                optimizer.zero_grad() # Очищаем градиенты для следующего шага аккумуляции
            
            train_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
            preds = outputs.argmax(dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_true.extend(labels.cpu().numpy())
            pbar.set_postfix(loss=f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}")

        # ==================== ЭТАП ВАЛИДАЦИИ (VALIDATION) ====================
        model.eval()
        val_loss = 0.0
        val_preds, val_true = [], []
        
        with torch.no_grad():
            pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1:2d}/{num_epochs} [Val  ]", leave=False)
            for images, labels in pbar_val:
                if images.numel() == 0: continue
                images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
                
                with autocast(dtype=torch.float16):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                preds = outputs.argmax(dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        
        # ==================== РАСЧЕТ МЕТРИК И ВЫВОД ====================
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_f1 = f1_score(train_true, train_preds, average='binary', zero_division=0)
        val_f1 = f1_score(val_true, val_preds, average='binary', zero_division=0)
        val_acc = accuracy_score(val_true, val_preds)
        
        print(f"Epoch {epoch+1:2d}/{num_epochs} -> "
              f"Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f} | "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")

        # ==================== СОХРАНЕНИЕ ЛУЧШЕЙ МОДЕЛИ И EARLY STOPPING ====================
        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            save_path = f"{MODEL_SAVE_DIR}/best_model_epoch_{epoch+1}_f1_{val_f1:.4f}.pth"
            torch.save(best_model_state, save_path)
            print(f"✅ Best F1 score! Model saved to {save_path}")
        else:
            patience_counter += 1
            
        if patience_counter >= 7:
            print(f"🛑 Early stopping triggered after {patience_counter} epochs without improvement.")
            break
            
        scheduler.step(val_f1) # Планировщик реагирует на F1-меру на валидации

    # Загружаем веса лучшей модели перед возвратом
    if best_model_state:
        model.load_state_dict(best_model_state)
    return model


# ========================== MAIN ==========================
if __name__ == '__main__':
    model = DeepfakeCNN().to(DEVICE)
    
    # ⚠️ ИСПРАВЛЕНИЕ 3: Компиляция модели для ускорения (для PyTorch 2.0+)
    # Дает значительный прирост производительности "из коробки"
    
    print("\n" + "="*50)
    print("🚀 Начинаем оптимизированное обучение с обработкой батчами...")
    print("="*50)

    trained_model = train_model(
        model, 
        train_loader, 
        val_loader, 
        class_weights,
        num_epochs=70
    )

    # Сохранение финальной лучшей модели
    torch.save(trained_model.state_dict(), "models/final_best_model.pth")
    print("\n✅ Final best model saved as 'models/final_best_model.pth'")

C:\Users\Иван\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Workers: 0, Batch Size: 32, Gradient Accumulation: 2, Effective Batch Size: 64

📊 Анализ дисбаланса и веса классов:
  Всего образцов: 50000
  Реальные лица (Класс 0): 41500 -> Weight: 0.60
  Синтетические лица (Класс 1): 8500 -> Weight: 2.94

>>> Проведение Oversampling для Fake класса...
--------------------------------------------------
Исходный размер обучающего набора: 40000 (R=33200, F=6800)
Балансированный размер обучающего набора: 66400 (R=33200, F=33200). Соотношение F/R ≈ 1.00

DataLoaders created successfully! Workers: 0
Train batches (Balanced): 2075, Val batches: 313


C:\Users\Иван\AppData\Local\Temp\ipykernel_17236\2532523361.py:236: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



🚀 Начинаем оптимизированное обучение с обработкой батчами...


Epoch  1/70 [Train]:   0%|          | 0/2075 [00:00<?, ?it/s]C:\Users\Иван\AppData\Local\Temp\ipykernel_17236\2532523361.py:262: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):


KeyboardInterrupt: 